In [0]:
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Setup MLflow Experiment
experiment_path = "/Workspace/Users/this15anmoldesu@gmail.com/Databricks 14 Day AI challenge - 02/DAY - 14"
mlflow.set_experiment(experiment_path)

# 2. Start Manual Run (No Autologging)
with mlflow.start_run(run_name="PlantGrowth_Manual_RF_Optimized") as run:
    
    # 3. Data Preparation
    df = spark.table("luffy.phase2.plant_growth_data")
    train, test = df.randomSplit([0.7, 0.3], seed=42)
    
    # 4. Define Pipeline Stages
    soil_indexer = StringIndexer(inputCol="Soil_Type", outputCol="soil_index", handleInvalid="keep")
    water_indexer = StringIndexer(inputCol="Water_Frequency", outputCol="water_index", handleInvalid="keep")
    fert_indexer = StringIndexer(inputCol="Fertilizer_Type", outputCol="fert_index", handleInvalid="keep")
    
    assembler = VectorAssembler(
        inputCols=["soil_index", "water_index", "fert_index", "Sunlight_Hours", "Temperature", "Humidity"],
        outputCol="features"
    )
    
    # 5. Random Forest with Manual Hyperparameters
    rf_trees = 24
    rf_depth = 10
    
    rf = RandomForestClassifier(
        featuresCol="features", 
        labelCol="Growth_Milestone",
        numTrees=rf_trees,
        maxDepth=rf_depth,
        seed=42
    )
    
    # 6. Fit Pipeline
    pipeline = Pipeline(stages=[soil_indexer, water_indexer, fert_indexer, assembler, rf])
    pipeline_model = pipeline.fit(train)
    
    # 7. Evaluation
    predictions = pipeline_model.transform(test)
    evaluator = MulticlassClassificationEvaluator(
        labelCol="Growth_Milestone", 
        predictionCol="prediction", 
        metricName="accuracy"
    )
    accuracy = evaluator.evaluate(predictions)
    
    # 8. MANUAL MLFLOW LOGGING
    # Log Parameters
    mlflow.log_param("model_type", "RandomForestClassifier")
    mlflow.log_param("num_trees", rf_trees)
    mlflow.log_param("max_depth", rf_depth)
    
    # Log Metrics
    mlflow.log_metric("accuracy", accuracy)
    
    # Log Model with Signature
    input_example = train.limit(5).toPandas()
    mlflow.spark.log_model(
        spark_model=pipeline_model, 
        artifact_path="random_forest_model",
        input_example=input_example,
        dfs_tmpdir="/Volumes/luffy/phase2/gold/tmp"
    )

    print(f"Manual Run Successful.")
    print(f"Test Accuracy: {accuracy:.2%}")
    print(f"Run ID: {run.info.run_id}")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/05 16:36:10 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==

Manual Run Successful.
Test Accuracy: 61.11%
Run ID: b514e8881b7b43debdb70d980644dc55
